# Unified FL Benchmark Pipeline — CSAI-4-CPS
Pipeline único e padronizado para Cumulative, Scaling, Sybil e OnOff.


In [2]:
# ==============================================================================
# UNIFIED FL BENCHMARK PIPELINE — CSAI-4-CPS
# ==============================================================================
# Executa e padroniza 4 cenários adversariais:
#   1) Cumulative / Cumulativo / Slow Drift / ASR Fixo
#   2) Scaling / Model Replacement
#   3) Sybil / Identity Flood
#   4) On-Off / Sleeping Cell
#
# Saídas padronizadas por dataset:
#   unified_results/<dataset>/<attack>/
#       raw_results.csv
#       summary_by_stage.csv
#       summary_by_strategy.csv
#       final_stage_ranking.csv
#       execution_report.csv
#       plots/F1_over_stage.png
#       plots/ASR_over_stage.png
#       plots/Time_over_stage.png
#
# Saídas consolidadas:
#   unified_results/all_raw_results.csv
#   unified_results/all_summary_by_stage.csv
#   unified_results/all_final_stage_ranking.csv
#   unified_results/all_diagnostic.csv
#   unified_results.zip
#
# Uso no Colab:
#   1. Ajuste ATTACKS_TO_RUN = ["Cumulative", "Scaling", "Sybil", "OnOff"] ou uma lista menor.
#   2. Execute a célula.
#   3. Faça upload dos datasets CSV curados.
# ==============================================================================

import subprocess
import sys
import time
import os
import io
import warnings
import shutil
import zipfile
import unicodedata
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import f1_score
from sklearn.metrics.pairwise import cosine_distances
from scipy.spatial.distance import cdist

try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

try:
    import hdbscan
except ImportError:
    print("Instalando dependência hdbscan...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "hdbscan", "-q"])
    import hdbscan

warnings.filterwarnings("ignore")

# ==============================================================================
# 1. CONFIGURAÇÕES GERAIS
# ==============================================================================

ATTACKS_TO_RUN = ["Cumulative", "Scaling", "Sybil", "OnOff"]
# Exemplo para rodar apenas um:
# ATTACKS_TO_RUN = ["Scaling"]

N_ROUNDS_STATISTICAL = 3
INITIAL_SEED = 42

STRATEGIES = [
    "FedAvg",
    "Krum",
    "FLAME",
    "FLTrust",
    "Adaptive_Ultimate"
]

# Configurações do modelo
HIDDEN_LAYERS_DEFAULT = (64,)
HIDDEN_LAYERS_CUMULATIVE = (32,)
LOCAL_EPOCHS_DEFAULT = 3
LOCAL_EPOCHS_CUMULATIVE = 2

# Parâmetros comuns de ataque
TRIGGER_VALUE = 5.0
POISON_RATE = 0.5
BENIGN_CLASS = 0
TARGET_ATTACK_CLASS = 1

# Parâmetros específicos
SCALING_FACTOR = 100.0
DRIFT_EPSILON = 0.20
CUMULATIVE_ATTACK_START_STAGE = 1

# Parâmetros das defesas
ALPHA_PENALTY = 0.85
BETA_RECOVERY = 0.10
TRUST_THRESHOLD = 0.2
FLAME_NOISE_SIGMA = 0.001

# Saída
OUTPUT_ROOT = Path("unified_results")
OUTPUT_ROOT.mkdir(exist_ok=True)


# ==============================================================================
# 2. UTILITÁRIOS
# ==============================================================================

def normalize_text(s: str) -> str:
    s = str(s)
    s = unicodedata.normalize("NFKD", s)
    s = "".join(c for c in s if not unicodedata.combining(c))
    s = s.lower()
    s = s.replace("-", "_").replace(" ", "_")
    s = re.sub(r"_+", "_", s)
    return s


def safe_slug(s: str, max_len: int = 120) -> str:
    s = str(s).strip()
    s = re.sub(r"\s+", "_", s)
    s = re.sub(r"[^A-Za-z0-9_\-]+", "", s)
    return s[:max_len] if s else "unknown"


def detect_dataset_name(filename: str, df: pd.DataFrame = None) -> str:
    text = filename
    if df is not None and "Dataset" in df.columns and df["Dataset"].notna().any():
        text += " " + str(df["Dataset"].dropna().iloc[0])

    s = normalize_text(text)
    original = str(text)

    if "dataset_a" in s or "cps_netflow" in s or "cps-netflow" in original.lower():
        return "A_CPS-NetFlow-IDS"
    if "dataset_b" in s or "packet_level" in s or "packet-level" in original.lower() or "botnet" in s:
        return "B_Packet-Level Botnet Set"
    if "dataset_c" in s or "mqtt_iot" in s or "mqtt-iot" in original.lower() or "mqtt" in s:
        return "C_MQTT-IoT-Flood"
    if "dataset_d" in s or "aggregated_traffic" in s or "aggregated traffic" in original.lower():
        return "D_Aggregated Traffic Set"

    return Path(filename).stem


def find_target_column(df: pd.DataFrame) -> str:
    candidates = ["class", "target", "label", "attack", "benign"]
    for c in df.columns:
        if c.lower() in candidates:
            return c
    return df.columns[-1]


def preprocess_dataframe(df: pd.DataFrame):
    cols_ignore = [
        "Dataset_Source",
        "dataset_source",
        "device_category",
        "host",
        "ts",
        "date",
        "time",
        "timestamp",
        "source_csv",
        "capture_id",
        "usage",
        "round_id",
        "Attack_Type",
        "attack_type"
    ]

    df = df.drop(columns=[c for c in cols_ignore if c in df.columns], errors="ignore")

    target_col = find_target_column(df)

    if df[target_col].dtype == object:
        df[target_col] = LabelEncoder().fit_transform(df[target_col].astype(str))

    y_series = df[target_col].astype(int)

    df_x = (
        df.drop(columns=[target_col])
        .select_dtypes(include=[np.number])
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0)
    )

    clean_df = pd.concat([df_x, y_series], axis=1)

    return clean_df, target_col


def get_attack_config(attack_name: str):
    if attack_name == "Cumulative":
        return {
            "canonical": "Cumulative",
            "benchmark": "Ataque_Cumulativo_ASR_Fixo",
            "nodes": ["Node1", "Node2", "Node3", "Node4"],
            "stages": list(range(17)),
            "hidden_layers": HIDDEN_LAYERS_CUMULATIVE,
            "local_epochs": LOCAL_EPOCHS_CUMULATIVE
        }

    if attack_name == "Scaling":
        return {
            "canonical": "Scaling",
            "benchmark": "Ataque_ModelReplacement_Scaling",
            "nodes": ["Node1", "Node2", "Node3", "Node4"],
            "stages": list(range(17)),
            "hidden_layers": HIDDEN_LAYERS_DEFAULT,
            "local_epochs": LOCAL_EPOCHS_DEFAULT
        }

    if attack_name == "Sybil":
        nodes = [f"Honest_{i}" for i in range(3)] + [f"Sybil_{i}" for i in range(7)]
        return {
            "canonical": "Sybil",
            "benchmark": "Ataque_Sybil_IdentityFlood",
            "nodes": nodes,
            "sybil_indices": list(range(3, 10)),
            "stages": list(range(17)),
            "hidden_layers": HIDDEN_LAYERS_DEFAULT,
            "local_epochs": LOCAL_EPOCHS_DEFAULT
        }

    if attack_name == "OnOff":
        return {
            "canonical": "OnOff",
            "benchmark": "Ataque_OnOff_SleepingCell",
            "nodes": ["Node1", "Node2", "Node3", "Node4"],
            "sleeping_cell_indices": [0],
            "stages": list(range(17)),
            "hidden_layers": HIDDEN_LAYERS_DEFAULT,
            "local_epochs": LOCAL_EPOCHS_DEFAULT
        }

    raise ValueError(f"Ataque desconhecido: {attack_name}")


# ==============================================================================
# 3. MODELO E FL
# ==============================================================================

def create_model(seed, hidden_layers):
    return MLPClassifier(
        hidden_layer_sizes=hidden_layers,
        activation="relu",
        solver="adam",
        random_state=seed,
        max_iter=1
    )


def get_weights(model):
    return np.concatenate(
        [w.flatten() for w in model.coefs_] +
        [b.flatten() for b in model.intercepts_]
    )


def set_weights(model, flat_weights, ref_model):
    idx = 0

    for i in range(len(model.coefs_)):
        shape = ref_model.coefs_[i].shape
        size = np.prod(shape)
        model.coefs_[i] = flat_weights[idx:idx + size].reshape(shape)
        idx += size

    for i in range(len(model.intercepts_)):
        shape = ref_model.intercepts_[i].shape
        size = np.prod(shape)
        model.intercepts_[i] = flat_weights[idx:idx + size].reshape(shape)
        idx += size


def local_training(global_weights, X, y, classes, ref_model, seed, hidden_layers, local_epochs):
    model = create_model(seed, hidden_layers)

    if len(X) >= 5:
        model.partial_fit(X[:5], y[:5], classes=classes)
    else:
        model.partial_fit(X, y, classes=classes)

    set_weights(model, global_weights.copy(), ref_model)

    rng = np.random.RandomState(seed)

    for _ in range(local_epochs):
        p = rng.permutation(len(X))
        model.partial_fit(X[p], y[p], classes=classes)

    return get_weights(model) - global_weights


# ==============================================================================
# 4. ATAQUE E AGREGAÇÃO
# ==============================================================================

def get_attack_state(attack_name, stage_idx, node_idx, config):
    nodes = config["nodes"]
    n_nodes = len(nodes)

    attack_active = False
    is_attacker = False
    active_attackers = 0
    attack_note = "inactive"

    if attack_name == "Cumulative":
        attack_active = stage_idx >= CUMULATIVE_ATTACK_START_STAGE
        is_attacker = node_idx == 0 and attack_active
        active_attackers = 1 if attack_active else 0
        attack_note = "cumulative_drift" if attack_active else "warmup"

    elif attack_name == "Scaling":
        active_attackers = 0
        if stage_idx > 0:
            active_attackers = min((stage_idx - 1) // 4 + 1, n_nodes)
        is_attacker = node_idx < active_attackers
        attack_active = active_attackers > 0
        attack_note = f"scaling_attackers_{active_attackers}"

    elif attack_name == "Sybil":
        attack_active = stage_idx > 0
        sybil_indices = config["sybil_indices"]
        is_attacker = node_idx in sybil_indices and attack_active
        active_attackers = len(sybil_indices) if attack_active else 0
        attack_note = "sybil_active" if attack_active else "warmup"

    elif attack_name == "OnOff":
        attack_active = (5 <= stage_idx <= 9) or (stage_idx >= 15)
        sleeping_indices = config["sleeping_cell_indices"]
        is_attacker = node_idx in sleeping_indices and attack_active
        active_attackers = len(sleeping_indices) if attack_active else 0
        attack_note = "onoff_active" if attack_active else "sleeping"

    return {
        "attack_active": attack_active,
        "is_attacker": is_attacker,
        "active_attackers": active_attackers,
        "attack_note": attack_note
    }


def apply_local_poisoning(X_n, y_n, seed, stage_idx, node_idx, target_class):
    n_p = int(len(y_n) * POISON_RATE)

    if n_p > 0:
        rng = np.random.RandomState(seed + stage_idx + node_idx)
        p_idx = rng.choice(len(y_n), n_p, replace=False)

        if X_n.shape[1] > 0:
            X_n[p_idx, -1] = TRIGGER_VALUE

        y_n[p_idx] = target_class

    return X_n, y_n


def aggregate_updates(strategy, local_updates, root_delta, root_norm, g_weights, trust_memory, nodes):
    agg_update = np.zeros_like(g_weights)

    if strategy == "FedAvg":
        return np.mean(local_updates, axis=0), trust_memory, 0, len(local_updates)

    if strategy == "Krum":
        k = max(1, len(local_updates) - 3)
        dists = cdist(np.stack(local_updates), np.stack(local_updates))
        scores = [np.sum(np.sort(d)[1:k + 1]) for d in dists]
        return local_updates[np.argmin(scores)], trust_memory, 0, 1

    if strategy == "FLAME":
        stk = np.stack(local_updates)
        dists = cosine_distances(stk)

        labels = hdbscan.HDBSCAN(
            metric="precomputed",
            min_cluster_size=2,
            allow_single_cluster=True
        ).fit_predict(dists)

        u, c = np.unique(labels, return_counts=True)
        valid = u[u != -1]

        if len(valid) > 0:
            valid_counts = {label: np.sum(labels == label) for label in valid}
            best_label = max(valid_counts, key=valid_counts.get)
            sel = stk[labels == best_label]
        else:
            sel = stk

        norms = np.linalg.norm(sel, axis=1)
        med = np.median(norms) if len(norms) else 0.0

        clip = [
            v * min(1, med / (np.linalg.norm(v) + 1e-12))
            for v in sel
        ]

        agg_update = (
            np.mean(clip, axis=0) +
            np.random.normal(0, FLAME_NOISE_SIGMA * med, g_weights.shape)
        )

        return agg_update, trust_memory, 0, len(sel)

    if strategy == "FLTrust":
        scores = []

        for d in local_updates:
            sim = np.dot(root_delta, d) / (
                root_norm * (np.linalg.norm(d) + 1e-12)
            )

            score = max(0, sim)
            scores.append(score)

            norm_up = d * (
                root_norm / (np.linalg.norm(d) + 1e-12)
            )

            agg_update += score * norm_up

        if sum(scores) > 0:
            agg_update /= sum(scores)

        trusted_clients = int(np.sum(np.array(scores) > 0))

        return agg_update, trust_memory, 0, trusted_clients

    if strategy == "Adaptive_Ultimate":
        valid_cnt = 0
        maj_thr = (len(nodes) // 2) + 1
        trust_w = []

        for i, d in enumerate(local_updates):
            norm_factor = root_norm / (
                np.linalg.norm(d) + 1e-12
            )

            d_norm = d * norm_factor

            sim = np.dot(root_delta, d_norm) / (
                root_norm * (np.linalg.norm(d_norm) + 1e-12)
            )

            old = trust_memory[nodes[i]]
            raw_score = max(0, sim)

            if sim < 0:
                new = 0.0
            elif (raw_score - old) < 0:
                new = old + (ALPHA_PENALTY * (raw_score - old))
            else:
                new = old + (BETA_RECOVERY * (raw_score - old))

            new = np.clip(new, 0.0, 1.0)
            trust_memory[nodes[i]] = new

            if new < TRUST_THRESHOLD:
                w = 0.0
            else:
                w = new
                valid_cnt += 1

            trust_w.append(w)

        circuit_breaker = 0

        if valid_cnt < maj_thr:
            circuit_breaker = 1
            agg_update = np.zeros_like(g_weights)
        else:
            tot = 0.0

            for i, w in enumerate(trust_w):
                if w > 0:
                    norm_up = local_updates[i] * (
                        root_norm / (np.linalg.norm(local_updates[i]) + 1e-12)
                    )

                    agg_update += w * norm_up
                    tot += w

            if tot > 0:
                agg_update /= tot

        return agg_update, trust_memory, circuit_breaker, valid_cnt

    raise ValueError(f"Estratégia desconhecida: {strategy}")


# ==============================================================================
# 5. EXECUÇÃO DE UM ATAQUE
# ==============================================================================

def run_single_seed(df, target_col, seed, dataset_name, attack_name):
    config = get_attack_config(attack_name)

    nodes = config["nodes"]
    stages = config["stages"]
    hidden_layers = config["hidden_layers"]
    local_epochs = config["local_epochs"]

    y_raw = df[target_col].values.astype(int)
    X_raw = df.drop(columns=[target_col]).values
    classes = np.unique(y_raw)

    X_train, X_test, y_train, y_test = train_test_split(
        X_raw,
        y_raw,
        test_size=0.2,
        stratify=y_raw,
        random_state=seed
    )

    X_root, X_eval, y_root, y_eval = train_test_split(
        X_test,
        y_test,
        test_size=0.5,
        stratify=y_test,
        random_state=seed
    )

    scaler = StandardScaler().fit(X_train)

    X_root_sc = scaler.transform(X_root)
    X_eval_sc = scaler.transform(X_eval)

    # Backdoor test set: benign samples + trigger
    mask_benign = (y_eval == BENIGN_CLASS)

    if np.sum(mask_benign) > 0:
        X_backdoor = X_eval_sc[mask_benign].copy()
        if X_backdoor.shape[1] > 0:
            X_backdoor[:, -1] = TRIGGER_VALUE
    else:
        X_backdoor = np.empty((0, X_eval_sc.shape[1]))

    results = []

    print(f"\n   Seed {seed} | Attack={attack_name} | Dataset={dataset_name}")

    for strategy in STRATEGIES:
        strategy_start_time = time.perf_counter()

        g_model = create_model(seed, hidden_layers)
        g_model.partial_fit(X_eval_sc[:10], y_eval[:10], classes=classes)

        g_weights = get_weights(g_model)
        trust_memory = {n: 0.5 for n in nodes}

        for stage_idx in stages:
            stage_start_time = time.perf_counter()
            stage_name = f"S{stage_idx}"

            shards_X = np.array_split(X_train, len(nodes))
            shards_y = np.array_split(y_train, len(nodes))

            local_updates = []
            active_attackers_stage = 0
            attack_active_stage = False
            attack_note = "inactive"

            for i, node in enumerate(nodes):
                X_n = shards_X[i].copy()
                y_n = shards_y[i].copy()

                attack_state = get_attack_state(attack_name, stage_idx, i, config)

                if attack_state["attack_active"]:
                    attack_active_stage = True
                    active_attackers_stage = max(active_attackers_stage, attack_state["active_attackers"])
                    attack_note = attack_state["attack_note"]

                if attack_state["is_attacker"]:
                    X_n, y_n = apply_local_poisoning(
                        X_n,
                        y_n,
                        seed,
                        stage_idx,
                        i,
                        TARGET_ATTACK_CLASS
                    )

                delta = local_training(
                    g_weights,
                    scaler.transform(X_n),
                    y_n,
                    classes,
                    g_model,
                    seed,
                    hidden_layers,
                    local_epochs
                )

                # Specific post-processing of malicious updates
                if attack_state["is_attacker"] and attack_name == "Scaling":
                    delta = delta * SCALING_FACTOR

                if attack_state["is_attacker"] and attack_name == "Cumulative":
                    delta = delta * DRIFT_EPSILON

                local_updates.append(delta)

            root_delta = local_training(
                g_weights,
                X_root_sc,
                y_root,
                classes,
                g_model,
                seed,
                hidden_layers,
                local_epochs
            )

            root_norm = np.linalg.norm(root_delta) + 1e-12

            agg_update, trust_memory, circuit_breaker, trusted_clients = aggregate_updates(
                strategy=strategy,
                local_updates=local_updates,
                root_delta=root_delta,
                root_norm=root_norm,
                g_weights=g_weights,
                trust_memory=trust_memory,
                nodes=nodes
            )

            g_weights += agg_update
            set_weights(g_model, g_weights, g_model)

            y_pred = g_model.predict(X_eval_sc)

            f1 = f1_score(
                y_eval,
                y_pred,
                average="macro"
            )

            asr = (
                np.mean(g_model.predict(X_backdoor) == TARGET_ATTACK_CLASS)
                if len(X_backdoor) > 0
                else 0.0
            )

            stage_time = time.perf_counter() - stage_start_time
            cumulative_strategy_time = time.perf_counter() - strategy_start_time

            results.append({
                "Benchmark": config["benchmark"],
                "Attack": attack_name,
                "Dataset": dataset_name,
                "Seed": seed,
                "Strategy": strategy,
                "Stage": stage_idx,
                "Stage_Name": stage_name,
                "Attack_Active": int(attack_active_stage),
                "Attack_Note": attack_note,
                "Active_Attackers": active_attackers_stage,
                "Total_Nodes": len(nodes),
                "Trusted_Clients": trusted_clients,
                "Circuit_Breaker": circuit_breaker,
                "F1": f1,
                "ASR": asr,
                "Time": stage_time,
                "Cumulative_Time_Strategy": cumulative_strategy_time
            })

        final = results[-1]

        print(
            f"      {strategy.ljust(17)} | "
            f"F1={final['F1']:.4f} | "
            f"ASR={final['ASR']:.4f} | "
            f"Time={final['Time']:.4f}s | "
            f"CumTime={final['Cumulative_Time_Strategy']:.4f}s"
        )

    return results


# ==============================================================================
# 6. SUMARIZAÇÃO E GRÁFICOS
# ==============================================================================

def save_outputs(full_df: pd.DataFrame, out_dir: Path):
    out_dir.mkdir(parents=True, exist_ok=True)
    plots_dir = out_dir / "plots"
    plots_dir.mkdir(exist_ok=True)

    full_df.to_csv(out_dir / "raw_results.csv", index=False)

    summary_by_stage = (
        full_df
        .groupby(["Benchmark", "Attack", "Dataset", "Strategy", "Stage"], as_index=False)
        .agg(
            F1_Mean=("F1", "mean"),
            F1_Std=("F1", "std"),
            ASR_Mean=("ASR", "mean"),
            ASR_Std=("ASR", "std"),
            Time_Mean=("Time", "mean"),
            Time_Std=("Time", "std"),
            Cumulative_Time_Strategy_Mean=("Cumulative_Time_Strategy", "mean"),
            Cumulative_Time_Strategy_Std=("Cumulative_Time_Strategy", "std"),
            Attack_Active=("Attack_Active", "max"),
            Active_Attackers=("Active_Attackers", "max"),
            Trusted_Clients_Mean=("Trusted_Clients", "mean"),
            Circuit_Breaker_Rate=("Circuit_Breaker", "mean")
        )
    )

    for c in ["F1_Std", "ASR_Std", "Time_Std", "Cumulative_Time_Strategy_Std"]:
        summary_by_stage[c] = summary_by_stage[c].fillna(0)

    summary_by_stage.to_csv(out_dir / "summary_by_stage.csv", index=False)

    summary_by_strategy = (
        full_df
        .groupby(["Benchmark", "Attack", "Dataset", "Strategy"], as_index=False)
        .agg(
            F1_Global_Mean=("F1", "mean"),
            F1_Global_Std=("F1", "std"),
            ASR_Global_Mean=("ASR", "mean"),
            ASR_Global_Std=("ASR", "std"),
            Time_Global_Mean=("Time", "mean"),
            Time_Global_Std=("Time", "std"),
            Total_Time_Strategy_Max=("Cumulative_Time_Strategy", "max"),
            Circuit_Breaker_Rate=("Circuit_Breaker", "mean")
        )
    )

    summary_by_strategy.to_csv(out_dir / "summary_by_strategy.csv", index=False)

    final_rows = []
    for (_, _, _, strategy), g in summary_by_stage.groupby(["Attack", "Dataset", "Benchmark", "Strategy"]):
        max_stage = g["Stage"].max()
        final_rows.append(g[g["Stage"] == max_stage])

    final_stage = pd.concat(final_rows, ignore_index=True)

    final_stage = final_stage.sort_values(
        ["Attack", "Dataset", "ASR_Mean", "F1_Mean"],
        ascending=[True, True, True, False]
    )

    final_stage.to_csv(out_dir / "final_stage_ranking.csv", index=False)

    # Plots
    for metric, std_col, ylabel, ylim in [
        ("F1_Mean", "F1_Std", "F1-score", (-0.02, 1.02)),
        ("ASR_Mean", "ASR_Std", "ASR", (-0.02, 1.02)),
        ("Time_Mean", "Time_Std", "Time (seconds)", None)
    ]:
        plt.figure(figsize=(10, 5), dpi=180)

        for strategy, g in summary_by_stage.groupby("Strategy"):
            g = g.sort_values("Stage")
            x = g["Stage"].values
            y = g[metric].values
            std = g[std_col].fillna(0).values

            plt.plot(
                x,
                y,
                marker="o",
                linewidth=1.8,
                markersize=3,
                label=strategy
            )

            plt.fill_between(
                x,
                y - std,
                y + std,
                alpha=0.12
            )

        attack = full_df["Attack"].iloc[0]
        dataset = full_df["Dataset"].iloc[0]

        plt.title(f"{attack} — {dataset}\nMean {ylabel} over stages")
        plt.xlabel("Stage")
        plt.ylabel(ylabel)
        plt.grid(True, alpha=0.25)
        plt.xticks(sorted(summary_by_stage["Stage"].unique()))

        if ylim:
            plt.ylim(*ylim)

        plt.legend(
            loc="center left",
            bbox_to_anchor=(1.02, 0.5),
            frameon=False,
            fontsize=9
        )

        plt.tight_layout()
        plt.savefig(plots_dir / f"{metric.replace('_Mean', '')}_over_stage.png", bbox_inches="tight")
        plt.close()

    return summary_by_stage, summary_by_strategy, final_stage


def save_global_outputs(all_raw: pd.DataFrame):
    OUTPUT_ROOT.mkdir(exist_ok=True)

    all_raw.to_csv(OUTPUT_ROOT / "all_raw_results.csv", index=False)

    all_stage = (
        all_raw
        .groupby(["Benchmark", "Attack", "Dataset", "Strategy", "Stage"], as_index=False)
        .agg(
            F1_Mean=("F1", "mean"),
            F1_Std=("F1", "std"),
            ASR_Mean=("ASR", "mean"),
            ASR_Std=("ASR", "std"),
            Time_Mean=("Time", "mean"),
            Time_Std=("Time", "std"),
            Attack_Active=("Attack_Active", "max"),
            Active_Attackers=("Active_Attackers", "max"),
            Trusted_Clients_Mean=("Trusted_Clients", "mean"),
            Circuit_Breaker_Rate=("Circuit_Breaker", "mean")
        )
    )

    all_stage.to_csv(OUTPUT_ROOT / "all_summary_by_stage.csv", index=False)

    final_rows = []

    for (attack, dataset, strategy), g in all_stage.groupby(["Attack", "Dataset", "Strategy"]):
        max_stage = g["Stage"].max()
        final_rows.append(g[g["Stage"] == max_stage])

    all_final = pd.concat(final_rows, ignore_index=True)

    all_final = all_final.sort_values(
        ["Attack", "Dataset", "ASR_Mean", "F1_Mean"],
        ascending=[True, True, True, False]
    )

    all_final.to_csv(OUTPUT_ROOT / "all_final_stage_ranking.csv", index=False)

    diagnostic = (
        all_raw
        .groupby(["Attack", "Dataset"], as_index=False)
        .agg(
            rows=("F1", "count"),
            seeds=("Seed", "nunique"),
            strategies=("Strategy", "nunique"),
            min_stage=("Stage", "min"),
            max_stage=("Stage", "max"),
            total_nodes=("Total_Nodes", "max"),
            mean_time=("Time", "mean"),
            total_time=("Time", "sum")
        )
    )

    diagnostic.to_csv(OUTPUT_ROOT / "all_diagnostic.csv", index=False)

    return all_stage, all_final, diagnostic


# ==============================================================================
# 7. MAIN
# ==============================================================================

def main():
    print("UNIFIED FL BENCHMARK PIPELINE — CSAI-4-CPS")
    print("Ataques selecionados:", ATTACKS_TO_RUN)

    if IN_COLAB:
        print("\nFaça o upload dos datasets CSV curados.")
        uploaded = files.upload()

        if not uploaded:
            print("Nenhum arquivo enviado.")
            return

        datasets = []

        for filename, content in uploaded.items():
            df = pd.read_csv(io.BytesIO(content))
            dataset_name = detect_dataset_name(filename, df)
            clean_df, target_col = preprocess_dataframe(df)

            datasets.append({
                "filename": filename,
                "dataset_name": dataset_name,
                "df": clean_df,
                "target_col": target_col
            })
    else:
        # Execução local opcional:
        # Coloque CSVs em ./datasets
        dataset_dir = Path("datasets")
        datasets = []

        for csv_path in dataset_dir.glob("*.csv"):
            df = pd.read_csv(csv_path)
            dataset_name = detect_dataset_name(csv_path.name, df)
            clean_df, target_col = preprocess_dataframe(df)

            datasets.append({
                "filename": csv_path.name,
                "dataset_name": dataset_name,
                "df": clean_df,
                "target_col": target_col
            })

        if not datasets:
            print("Nenhum dataset encontrado em ./datasets.")
            return

    all_results = []

    for ds in datasets:
        dataset_name = ds["dataset_name"]
        df = ds["df"]
        target_col = ds["target_col"]

        print("\n" + "=" * 90)
        print(f"DATASET: {dataset_name}")
        print(f"Arquivo: {ds['filename']}")
        print(f"Linhas: {len(df)} | Features numéricas: {df.shape[1] - 1}")
        print("=" * 90)

        for attack in ATTACKS_TO_RUN:
            attack_start = time.perf_counter()

            attack_results = []

            for i in range(N_ROUNDS_STATISTICAL):
                seed = INITIAL_SEED + i

                attack_results.extend(
                    run_single_seed(
                        df=df,
                        target_col=target_col,
                        seed=seed,
                        dataset_name=dataset_name,
                        attack_name=attack
                    )
                )

            attack_total_time = time.perf_counter() - attack_start

            attack_df = pd.DataFrame(attack_results)

            out_dir = OUTPUT_ROOT / safe_slug(dataset_name) / attack
            summary_stage, summary_strategy, final_stage = save_outputs(attack_df, out_dir)

            execution_report = pd.DataFrame([{
                "Dataset": dataset_name,
                "Attack": attack,
                "Benchmark": get_attack_config(attack)["benchmark"],
                "Seeds": N_ROUNDS_STATISTICAL,
                "Strategies": len(STRATEGIES),
                "Stages": len(get_attack_config(attack)["stages"]),
                "Rows": len(df),
                "Features": df.shape[1] - 1,
                "Total_Execution_Time_Seconds": attack_total_time,
                "Total_Execution_Time_Minutes": attack_total_time / 60.0
            }])

            execution_report.to_csv(out_dir / "execution_report.csv", index=False)

            print(
                f"\nConcluído: {attack} — {dataset_name} | "
                f"Tempo total: {attack_total_time:.2f}s "
                f"({attack_total_time / 60.0:.2f} min)"
            )

            all_results.append(attack_df)

    if all_results:
        all_raw = pd.concat(all_results, ignore_index=True)

        all_stage, all_final, diagnostic = save_global_outputs(all_raw)

        print("\nResumo diagnóstico:")
        display(diagnostic)

        print("\nRanking final:")
        display(all_final.head(40))

        zip_name = "unified_results.zip"

        if Path(zip_name).exists():
            Path(zip_name).unlink()

        shutil.make_archive("unified_results", "zip", OUTPUT_ROOT)

        print(f"\nArquivo gerado: {zip_name}")

        if IN_COLAB:
            files.download(zip_name)


if __name__ == "__main__":
    main()


UNIFIED FL BENCHMARK PIPELINE — CSAI-4-CPS
Ataques selecionados: ['Cumulative', 'Scaling', 'Sybil', 'OnOff']

Faça o upload dos datasets CSV curados.


Saving Dataset_A_CPS-NetFlow-IDS.csv to Dataset_A_CPS-NetFlow-IDS (1).csv
Saving Dataset_B_Packet-Level Botnet Set.csv to Dataset_B_Packet-Level Botnet Set (1).csv
Saving Dataset_C_MQTT-IoT-Flood.csv to Dataset_C_MQTT-IoT-Flood (1).csv
Saving Dataset_D_Aggregated Traffic Set.csv to Dataset_D_Aggregated Traffic Set (1).csv

DATASET: A_CPS-NetFlow-IDS
Arquivo: Dataset_A_CPS-NetFlow-IDS (1).csv
Linhas: 2556 | Features numéricas: 58

   Seed 42 | Attack=Cumulative | Dataset=A_CPS-NetFlow-IDS
      FedAvg            | F1=0.7544 | ASR=0.7812 | Time=0.0503s | CumTime=0.9055s
      Krum              | F1=0.7460 | ASR=0.7812 | Time=0.0470s | CumTime=0.8687s
      FLAME             | F1=0.9504 | ASR=0.7812 | Time=0.0466s | CumTime=0.8638s
      FLTrust           | F1=0.6845 | ASR=0.7812 | Time=0.0491s | CumTime=0.8144s
      Adaptive_Ultimate | F1=0.6845 | ASR=0.7812 | Time=0.0485s | CumTime=0.8265s

   Seed 43 | Attack=Cumulative | Dataset=A_CPS-NetFlow-IDS
      FedAvg            | F1=0.9375 |

,Attack,Dataset,rows,seeds,strategies,min_stage,max_stage,total_nodes,mean_time,total_time
0,Cumulative,A_CPS-NetFlow-IDS,255,3,5,0,16,4,0.052876,13.483474
1,Cumulative,B_Packet-Level Botnet Set,255,3,5,0,16,4,0.044649,11.385479
2,Cumulative,C_MQTT-IoT-Flood,255,3,5,0,16,4,0.043961,11.210057
3,Cumulative,D_Aggregated Traffic Set,255,3,5,0,16,4,0.039775,10.142691
4,OnOff,A_CPS-NetFlow-IDS,255,3,5,0,16,4,0.117625,29.994320
5,OnOff,B_Packet-Level Botnet Set,255,3,5,0,16,4,0.057714,14.717002
6,OnOff,C_MQTT-IoT-Flood,255,3,5,0,16,4,0.057733,14.721792
7,OnOff,D_Aggregated Traffic Set,255,3,5,0,16,4,0.047548,12.124643
8,Scaling,A_CPS-NetFlow-IDS,255,3,5,0,16,4,0.116835,29.792984
9,Scaling,B_Packet-Level Botnet Set,255,3,5,0,16,4,0.057716,14.717568



Ranking final:


,Benchmark,Attack,Dataset,Strategy,Stage,F1_Mean,F1_Std,ASR_Mean,ASR_Std,Time_Mean,Time_Std,Attack_Active,Active_Attackers,Trusted_Clients_Mean,Circuit_Breaker_Rate
1,Ataque_Cumulativo_ASR_Fixo,Cumulative,A_CPS-NetFlow-IDS,FLAME,16,0.940460,0.015117,0.552083,0.396928,0.047860,0.002477,1,1,2.666667,0.000000
3,Ataque_Cumulativo_ASR_Fixo,Cumulative,A_CPS-NetFlow-IDS,FedAvg,16,0.879937,0.108848,0.552083,0.396928,0.054855,0.011730,1,1,4.000000,0.000000
4,Ataque_Cumulativo_ASR_Fixo,Cumulative,A_CPS-NetFlow-IDS,Krum,16,0.872320,0.110118,0.552083,0.396928,0.054708,0.014496,1,1,1.000000,0.000000
0,Ataque_Cumulativo_ASR_Fixo,Cumulative,A_CPS-NetFlow-IDS,Adaptive_Ultimate,16,0.728755,0.047903,0.598958,0.443137,0.047013,0.002578,1,1,3.666667,0.000000
2,Ataque_Cumulativo_ASR_Fixo,Cumulative,A_CPS-NetFlow-IDS,FLTrust,16,0.728755,0.047903,0.598958,0.443137,0.047513,0.001576,1,1,4.000000,0.000000
9,Ataque_Cumulativo_ASR_Fixo,Cumulative,B_Packet-Level Botnet Set,Krum,16,0.631777,0.176527,0.340000,0.571664,0.047299,0.016883,1,1,1.000000,0.000000
6,Ataque_Cumulativo_ASR_Fixo,Cumulative,B_Packet-Level Botnet Set,FLAME,16,0.684481,0.085825,0.666667,0.577350,0.042268,0.004073,1,1,2.666667,0.000000
8,Ataque_Cumulativo_ASR_Fixo,Cumulative,B_Packet-Level Botnet Set,FedAvg,16,0.684481,0.085825,0.673333,0.565803,0.046562,0.013494,1,1,4.000000,0.000000
5,Ataque_Cumulativo_ASR_Fixo,Cumulative,B_Packet-Level Botnet Set,Adaptive_Ultimate,16,0.682635,0.083836,0.673333,0.565803,0.046261,0.012367,1,1,4.000000,0.000000
7,Ataque_Cumulativo_ASR_Fixo,Cumulative,B_Packet-Level Botnet Set,FLTrust,16,0.682635,0.083836,0.673333,0.565803,0.041890,0.005725,1,1,4.000000,0.000000



Arquivo gerado: unified_results.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>